# v39 — Full T12: сила графа, без обучения

**Run All**: проверки → загрузка сохранённых признаков v38 → четыре calibration-варианта → фиксация победителя → validation победителя и v25 → отчёт.

Full T12 остаётся тем же: 925 train-ID, final step 1800, доля 10%. Сравниваем λ=0,50, λ=0,75 и raw без графа; контроль — текущий MVP_fusion_v25. k1=20/k2=3 не меняются.

Всё считается на CPU по готовым эмбеддингам. Ни обучения, ни повторного извлечения признаков, ни загрузки новых весов нет. Кандидат, confidence и порог отказа остаются от v25. Автоматического переключения MVP нет.

Calibration и validation уже наблюдались: это development-эксперимент, не независимый тест. Выбор только по calibration mAP@10; при точном равенстве — контроль, затем λ=.50, λ=.75, raw. Ограничения времени нет.

In [1]:
import os
os.environ['ORT_DISABLE_TELEMETRY'] = '1'  # до любых runtime-импортов

from pathlib import Path
import sys, subprocess, time

RUN_NAME = 'graph_v1'  # то же имя = проверенное продолжение
ROOT = next((p for root in [Path.cwd(), *Path.cwd().parents]
             for p in (root, root / 'Car-classification-MSK')
             if (p / 'training/transreid_graph.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Открой notebook внутри Car-classification-MSK')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
STARTED = time.perf_counter()
print('Python:', sys.executable)
print('Project:', ROOT)
print('CPU, cached features only | Run:', RUN_NAME, '| time limit: NONE')


Python: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python
Project: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK
CPU, cached features only | Run: graph_v1 | time limit: NONE


In [2]:
print('Этап 1/4: проверки графа, кандидатов, изоляции query и выбора по calibration')
subprocess.run([sys.executable, '-m', 'pytest', '-q',
    'tests/test_transreid_graph.py', 'tests/test_transreid_system.py',
    'tests/test_dual_role_inference.py', 'tests/test_retrieval_policy.py'],
    check=True, cwd=ROOT, env={**os.environ, 'ORT_DISABLE_TELEMETRY': '1'})
from training import transreid_graph as experiment


Этап 1/4: проверки графа, кандидатов, изоляции query и выбора по calibration
........................................................................ [ 61%]
..............................................                           [100%]
118 passed in 6.93s


In [3]:
print('Этап 2/4: проверка v38, весов, исходных данных и порядка признаков; хеширование может занять время')
context = experiment.prepare(run_name=RUN_NAME)
print('Output:', context['output'])
print('Signature:', context['signature'])
print('Frozen candidate threshold:', context['manifest']['threshold'])
print('Systems:', list(experiment.SYSTEMS))
print('Reviewed metadata transitions:', context['manifest']['reviewed_document_transition'])


Этап 2/4: проверка v38, весов, исходных данных и порядка признаков; хеширование может занять время
PREFLIGHT: verify v38 caches, weights, original data and two reviewed metadata transitions
CHECK: 13332 protected files; old manifests remain unchanged
PREFLIGHT OK: 4 calibration cases; CPU caches only; validation closed until selection
Output: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_39_transreid_graph/runs/graph_v1
Signature: b77923ecbd74869e682ec01bd6a1fd6aee4b13c1f38a010e37a4adaafc76ece2
Frozen candidate threshold: 0.534365177154541
Systems: ['V25_control', 'Full_T12_l50', 'Full_T12_l75', 'Full_T12_raw']
Reviewed metadata transitions: {'files': [{'path': '/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK-main/models/README.md', 'old_sha256': '2b34a64165009d38d963a82f847227b51905f7011a9d46a221ce88cd08b273c9', 'new_sha256': 'b58775744653ead7f5f5f53faa5d56aeb22cf88ce559fbfc275

Новый manifest допускает два точных проверенных metadata-перехода: описание существующего v25 в `main/models/README.md` и обновление списка требований в `main/release_decision.json` после уточнения организаторов (отмена требования extractor.py). Модель не менялась. Зафиксированы точные старые и новые hashes. Старые manifests и результаты не переписываются; остальные веса, данные и код по-прежнему проверяются. Любое дальнейшее изменение остановит запуск — не отключай проверку ради продолжения.

In [4]:
print('Этап 3/4: calibration × 4 → frozen selection → validation победителя и v25 → экспорт')
results = experiment.run(context, allow_outer=True)
assert results['status'] == 'complete' and results['candidate_unchanged']
assert results['optimizer_updates'] == results['encoder_forwards'] == results['inference_bn_updates'] == 0
assert not results['promoted'] and not results['threshold_fit']
assert set(results['evaluations']['validation']) == set(results['selection']['evaluations'])


Этап 3/4: calibration × 4 → frozen selection → validation победителя и v25 → экспорт
CHECK: 13332 protected files; old manifests remain unchanged
CALIBRATION 1/4: V25_control
STAGE calibration_V25_control: start
EVAL calibration/V25_control: mAP@10=0.826410; candidates unchanged
STAGE calibration_V25_control: complete in 1.2s
CALIBRATION 2/4: Full_T12_l50
STAGE calibration_Full_T12_l50: start
RANK T12_w10: 32/307 queries
RANK T12_w10: 64/307 queries
RANK T12_w10: 96/307 queries
RANK T12_w10: 128/307 queries
RANK T12_w10: 160/307 queries
RANK T12_w10: 192/307 queries
RANK T12_w10: 224/307 queries
RANK T12_w10: 256/307 queries
RANK T12_w10: 288/307 queries
RANK T12_w10: 307/307 queries
EVAL calibration/Full_T12_l50: mAP@10=0.829879; candidates unchanged
STAGE calibration_Full_T12_l50: complete in 1.3s
CALIBRATION 3/4: Full_T12_l75
STAGE calibration_Full_T12_l75: start
RANK Full_T12_l75: 307/307 queries
EVAL calibration/Full_T12_l75: mAP@10=0.828373; candidates unchanged
STAGE calibration

In [5]:
from IPython.display import Markdown, display

print('Этап 4/4: итог')
print('Победитель calibration:', results['selection']['selected'])
print('MVP_fusion_v25 остался активным.')
print(f'Текущий запуск notebook: {(time.perf_counter()-STARTED)/60:.1f} мин; это не benchmark модели.')
print('Report:', context['output'] / 'REPORT.md')
print('Results:', context['output'] / 'results.json')
display(Markdown((context['output'] / 'REPORT.md').read_text(encoding='utf-8')))


Этап 4/4: итог
Победитель calibration: Full_T12_l50
MVP_fusion_v25 остался активным.
Текущий запуск notebook: 1.3 мин; это не benchmark модели.
Report: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_39_transreid_graph/runs/graph_v1/REPORT.md
Results: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_39_transreid_graph/runs/graph_v1/results.json


# v39 — граф для сохранённого Full T12

Победитель calibration: **Full_T12_l50**. MVP_fusion_v25 не изменён.

| Split | Система | mAP@10 | Δ к v25, п.п. | Rank-1 | F1 | TNR |
|---|---|---:|---:|---:|---:|---:|
| calibration | V25_control | 0.826410 | +0.0000 | 0.841463 | 0.813483 | 0.704918 |
| calibration | Full_T12_l50 | 0.829879 | +0.3469 | 0.837398 | 0.813483 | 0.704918 |
| calibration | Full_T12_l75 | 0.828373 | +0.1963 | 0.837398 | 0.813483 | 0.704918 |
| calibration | Full_T12_raw | 0.807573 | -1.8837 | 0.829268 | 0.813483 | 0.704918 |
| validation | V25_control | 0.828957 | +0.0000 | 0.829960 | 0.795455 | 0.709677 |
| validation | Full_T12_l50 | 0.825002 | -0.3955 | 0.821862 | 0.795455 | 0.709677 |

Validation AP: улучшилось 15, ухудшилось 22, без изменения 210.
Top-1 исправлен у 2, испорчен у 4; top-10 изменился у 279 запросов.

## Что было зафиксировано

Один сохранённый Full T12 (925 train-ID, step 1800), доля 10%; 90% — прежняя смесь v25.
Только четыре calibration-варианта: контроль v25, Full T12 λ=.50, λ=.75, raw без графа. k1=20/k2=3.
Выбор по calibration mAP@10; точные равенства — в указанном порядке, контроль первый.
Validation рассчитана только для v25 и одного уже зафиксированного победителя; при победе контроля — один расчёт.
Кандидат/confidence/отказ — прежний R1 raw_top1; порог не меняется. candidates.csv совпадает с v25 побайтно.
Результаты старых λ=.50 и raw воспроизведены точно; NPY replay проверяет top-10, кандидатов, confidence и отказы.
Все эмбеддинги взяты из v38: 2048D для контроля, 2432D для Full T12. Новых encoder-forward, обучения, BN-update и threshold-fit нет.
Экспорты и полный порядок векторов: tasks/<split>_<system>/export/. Три файла, десять разных ID на query, включая отказы.
Статическая gallery, независимые query; ни имена файлов, ни ID/camera/time не передаются ранжировщику.
Это очередной development-эксперимент: calibration/validation уже наблюдались, выбор гипотезы учитывает прошлые результаты.
Ни независимый тест, ни новая оценка скорости модели, ни основание для автоматического promotion здесь не заявляются.
Проверка неизменности всех унаследованных весов/данных/кода сохранена. Два точных metadata-перехода учтены только в новом manifest.
README: 2b34a64165009d38d963a82f847227b51905f7011a9d46a221ce88cd08b273c9 → b58775744653ead7f5f5f53faa5d56aeb22cf88ce559fbfc2754b94150544500 (описание уже действующего v25).
release_decision: c95695b3eb3613c1d436e4622647e32f925c7cad814b723105f2a13a632767be → f306cef11ad5c7951dde3b1a59e9a55e2e03ff7bd16f996ad59c95a067bdd5c9 (отмена extractor-check и ссылка на уточнение; модель та же).
Старые manifests/guards/runs не изменены. Другой или последующий metadata-переход не разрешён автоматически.
Изменения приложения/GPU в main вне эксперимента не включаются в его runtime. Активный MVP не переключается.
Новых данных, OCR, изменений bbox и номерной зоны нет. Ограничения junk/top-10 и остаточного номерного сигнала остаются открытыми.
Этапы публикуются атомарно; повторный Run All проверяет и переиспользует завершённые результаты. Ограничения времени нет.


Повторный **Run All** с тем же RUN_NAME проверяет и использует законченные этапы; завершённый эксперимент заново не считается. После прерывания повторится только незавершённый этап. Не удаляй manifests, не меняй порог, данные или старые веса для прохождения проверки.

После завершения пришли «подсчёты завершились». Анализировать будем `runs/graph_v1/results.json`, `REPORT.md` и точные изменения top-10 относительно v25. Если победителем calibration стал контроль, validation запускается только для него.